# NHL: анализ статистики команд

**Тестовое задание • 100 баллов**

Вы аналитик, которому передали готовую таблицу с исторической статистикой команд NHL.
Изучите данные, проверьте их, постройте графики и объясните свои наблюдения.
В работе оцениваются чтение данных, анализ, графики и обоснованность выводов.

Ориентир времени: 3-5 часов (знание хоккея не требуется).
Можно обращаться к документации и пользоваться вспомогательными инструментами.
Укажите, чем пользовались; отвечайте за правильность и понимание сданной работы.
Подтверждайте ответы результатами из своего ноутбука.

## Что сдавать

Выполненный IPynb с видимыми результатами и ответами разместите на платформе githib.com.
Укажите имя, дату и используемые инструменты укажите в README.
Файл `data/hockey_raw.csv` уже приложен к заданию. Исходный файл сохраняйте без изменений.
Для развёрнутого ответа обычно достаточно 3-6 содержательных предложений.
Каждый такой ответ должен ссылаться на ваши числа, строки или графики.

## Данные

Одна строка соответствует одной команде в одном сезоне. Значение `year` является
меткой сезона на сайте. Источник готовой таблицы:
[Scrape This Site - Hockey Teams](https://www.scrapethissite.com/pages/forms/).
Работайте с приложенным CSV; обращения к сайту для выполнения задания не нужны.

| Столбец | Значение |
|---|---|
| `team` | Название команды |
| `year` | Метка сезона |
| `wins` | Победы |
| `losses` | Поражения |
| `ot_losses` | Поражения в дополнительное время, если значение указано |
| `win_rate` | Доля побед, указанная в источнике |
| `goals_for` | Забитые шайбы |
| `goals_against` | Пропущенные шайбы |
| `goal_diff` | Разница забитых и пропущенных шайб |

Пустое значение в `ot_losses` рассматривайте как пропуск.
Для графиков указывайте заголовок и подписи осей с единицами измерения.
При объяснении причин отделяйте то, что видно в таблице, от предположений.

## Баллы

| Раздел | Баллы |
|---|---:|
| 1. Загрузка и проверка | 25 |
| 2. Таблицы и группировка | 25 |
| 3. Графики | 30 |
| 4. Интерпретация | 20 |
| **Всего** | **100** |

Выполняйте пункты по порядку. Каждый пункт оценивается отдельно.

**Автор:** Сурова Юлия Леонидовна  
**Дата:** 13.10.2026.   

**Использованные источники и инструменты:** …

## 1.1. Загрузите таблицу: 5 баллов

Прочитайте приложенный CSV в pandas. Покажите первые пять строк,
        число строк и столбцов. Какие наблюдения представляет одна строка?

In [ ]:
import pandas as pd
df = pd.read_csv("hockey_raw.csv")
print(df.head())
print(df.shape)



**Ответ 1.1:**

В таблице 583 строки и 9 столбцов.Первые 5 строк показывают команды NHL за сезон 1990 года.Одна строка показывает одну команду в одном сезоне:об этом говорят столбцы team (название команды) и year (год сезона),и доп данные(голы, победы и тд).



## 1.2. Разберитесь со столбцами: 6 баллов

Выведите названия и типы столбцов. Посчитайте число разных команд и годов.
        Проверьте, повторяется ли пара `team` и `year`. Объясните результат проверки.

In [ ]:
import pandas as pd
df = pd.read_csv("hockey_raw.csv")

print(df.dtypes)#Названия и типы столбцов 
print("Число команд:", df["team"].nunique())
print("Число годов:", df["year"].nunique())

#Проверка повторов
print("Повторяющихся пар (team, year):",df.duplicated(subset=["team", "year"]).sum())


**Ответ 1.2:**

В таблице 9 столбцов:team (object),year (int64),wins, losses, ot_losses (float64, потому что есть пропуски),win_rate (float64), goals_for, goals_against, goal_diff (int64).Всего в таблице 35 разных команд и 21 разных годов.Пара "команда + год" повторяется 0 раз, то есть дубликатов нет

## 1.3. Найдите пропуски: 8 баллов

Посчитайте число и долю пропусков по столбцам. Для `ot_losses` покажите
        число пропусков по годам. Как будете обращаться с этим полем в дальнейшем анализе
        и почему?

In [ ]:
import pandas as pd
df = pd.read_csv("hockey_raw.csv")

print("Число пропусков по столбцам: ")
print(df.isna().sum())

print("\nДоля пропусков по столбцам:")
print(df.isna().mean())

print("\nПропуски в ot_losses по годам:")
print(df[df["ot_losses"].isna()].groupby("year").size())

**Ответ 1.3:**

Пропуски есть только в столбце ot_losses: 224 пропуска из 583 строк,то есть доля 38.488%. В других столбцах пропусков нет.Пропуски в ot_losses приходятся на 1990-1998 годы,в более поздних сезонах их нет.В дальнейшем анализе я буду обращаться с этим полем как с пропуском:не заполнять его средним или нулём, а учитывать отдельно, потому что отсутствие значения здесь означает не ошибку данных,а то что в эти годы показатель просто не фиксировался.Поэтому при расчётах, где нужен ot_losses,я буду использовать только строки с заполненным значением,а в описательной статистике — указывать число непустых наблюдений

## 1.4. Проверьте согласованность: 6 баллов

Проверьте, что `win_rate` лежит между 0 и 1, счётчики побед и голов
        неотрицательны, а `goal_diff` равен `goals_for - goals_against`.
        Сообщите, сколько строк нарушает каждое условие. Как бы вы поступили,
        если бы обнаружили нарушение?

In [ ]:
import pandas as pd
df = pd.read_csv("hockey_raw.csv")

bad_win_rate = ((df["win_rate"] < 0) | (df["win_rate"] > 1)).sum()
print("Строк с win_rate вне [0, 1]:", bad_win_rate)

cols = ["wins", "losses", "ot_losses", "goals_for", "goals_against"]
print("\nОтрицательные значения по столбцам:")
print((df[cols] < 0).sum())

bad_goal_diff = (df["goal_diff"] != df["goals_for"]- df["goals_against"]).sum()
print("\nСтрок, где goal_diff != goals_for - goals_against:", bad_goal_diff)

**Ответ 1.4:**

Проверка показала,что все условия выполняются:win_rate лежит в диапазоне [0,1] во 583 строках (то есть 0 нарушений),ни один из счётчиков (wins, losses, ot_losses, goals_for, goals_against) не содержит отрицательных значений (0 нарушений),и goal_diff равен goals_for-goals_against во всех строках (0 нарушений). Если бы нарушение обнаружилось, я бы сначала посмотрела на сами строки-нарушители,чтобы понять причину ошибки, и только потом решала бы исправить по правилу (пересчитать goal_diff) или исключить строку из анализа.

## 2.1. Опишите долю побед: 6 баллов

Найдите минимум, медиану, среднее и максимум `win_rate` по всей таблице.
        Сравните медиану и среднее. Что эти числа сообщают о типичном командном сезоне,
        а чего по ним узнать нельзя?

In [ ]:
import pandas as pd
df = pd.read_csv("hockey_raw.csv")

print("Минимум:", df["win_rate"].min())
print("Медиана:", df["win_rate"].median())
print("Среднее:", df["win_rate"].mean())
print("Максимум:",df["win_rate"].max())

**Ответ 2.1:**

По всей таблице win_rate имеет минимум: 0.119,медиану: 0.463, среднее: 0.4585189003436426 и максимум: 0.756.Медиана и среднее очень близки,разница примерно 0.005,что говорит о симметричном распределении без сильных выбросов.Типичный командный сезон -это примерно половина побед (win_rate около 0.5). По этим четырём числам нельзя узнать форму распределения, динамику по годам или конкретные команды-лидеры  для этого нужны гистограмма и группировки по годам

## 2.2. Сравните сезоны: 8 баллов

Для каждого года посчитайте число команд, медиану `wins` и медиану `win_rate`.
        Выведите таблицу целиком и отдельно сравните 1993, 1994 и 1995 годы.
        Как отличается картина по числу побед и по доле побед?

In [ ]:
import pandas as pd
df = pd.read_csv("hockey_raw.csv")

by_year = df.groupby("year").agg(
    teams=("team", "count"),
    median_wins=("wins", "median"),
    median_win_rate=("win_rate", "median")
)
print(by_year)
print(by_year.loc[[1993, 1994, 1995]])

**Ответ 2.2:**

Для каждого года посчитали число команд.Медиана wins и медиана win_rate.Сравнение 1993, 1994 и 1995 годов во всех трёх годах было по 26 команд, медиана побед составила: 39.0,20.5 и 35.5,а медиана win_rate :0.464, 0.4275 и 0.433.Год 1994 резко выделяется: медиана побед упала до 20.5 это сезон, сокращённый из за локаута ТК сыграно 48 матчей из 82, поэтому и число побед, и доля побед заметно ниже.Число побед отличается от доли побед - число побед растёт со временем (с 34 в 1990 до 41 в 2007-2011),потому что в лиге стало больше команд и матчей, тогда как доля побед остаётся стабильной в диапазоне примерно 0.42-0.52.Значит,для сравнения сезонов между собой лучше использовать win_rate,а не wins.

## 2.3. Найдите крайние результаты: 5 баллов

Покажите три строки с самой высокой и три с самой низкой `win_rate`.
        Укажите команду, год и значение. Как вы убедились, что сравниваете именно долю,
        а не число побед?

In [ ]:
import pandas as pd
df = pd.read_csv("hockey_raw.csv")

print("Три лучших по win_rate:")
print(df.nlargest(3, "win_rate")[["team","year", "win_rate", "wins"]])

print("\nТри худших по win_rate:")
print(df.nsmallest(3, "win_rate")[["team", "year", "win_rate", "wins"]])

**Ответ 2.3:**

Три строки с самой высокой win_rate занимает одна и та же команда: Detroit Red Wings (1995, win_rate =0.756),(2005, win_rate = 0.707),(1994, win_rate = 0.688).Три строки с самой низкой: Ottawa Senators (1992, win_rate = 0.119),San Jose Sharks(1992, win_rate = 0.131),Ottawa Senators(1993, win_rate = 0.167).Я убедилась,что сравниваю именно долю,а не число побед,тк сортировка выполнена по столбцу win_rate,а в выводе дополнительно показан столбец wins и видно, что команда с наибольшей долей побед не обязательно имеет наибольшее число побед,тк сезоны имеют разную длину.



## 2.4. Выберите команду: 6 баллов

Выберите команду, представленную не менее чем в пяти сезонах.
        Покажите её строки по годам и найдите сезон с минимальной и максимальной
        долей побед. Почему сравнивать только название команды без года неудобно?

In [ ]:
import pandas as pd
df = pd.read_csv("hockey_raw.csv")

team_counts = df["team"].value_counts()
print(team_counts.head(10))

team = team_counts.idxmax()
print("\nВыбранная команда:", team, "| сезонов:", team_counts.max())

team_df = df[df["team"] == team].sort_values("year")
team_df[["team", "year", "wins", "win_rate"]]

print("\nСезон с минимальной win_rate:")
print(team_df.loc[team_df["win_rate"].idxmin(), ["year", "win_rate"]])
print("\nСезон с максимальной win_rate:")
print(team_df.loc[team_df["win_rate"].idxmax(), ["year", "win_rate"]])

**Ответ 2.4:**

Я выбрала команду Boston Bruins,которая представлена в 21 сезоне.Сезон с минимальной долей побед: год (win_rate = 0.293), с максимальной — год (win_rate = 0.646).Сравнивать только название команды без года неудобно,тк одна команда участвует во многих сезонах с разными результатами и её сила меняется от сезона к сезону, и без года невозможно понять о каком именно периоде идёт речь.

## 3.1. Покажите распределение: 7 баллов

Постройте гистограмму `win_rate` для всех строк.
        Подпишите оси и объясните, в какой части шкалы расположено большинство наблюдений.
        Приведите числовое подтверждение из пункта 2.1.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv("hockey_raw.csv")

plt.hist(df["win_rate"], bins=20, edgecolor="black")
plt.title("Распределение win_rate по всем командам-сезонам")
plt.xlabel("win_rate (доля побед)")
plt.ylabel("Число команд-сезонов (шт.)")
plt.show()


**Ответ 3.1:**

Гистограмма win_rate имеет колоколообразную форму: большинство наблюдений расположено в центре шкалы,где то между 0.4 и 0.6, с пиком чуть меньше чем 0.5.Это подтверждается числами из пункта 2.1: медиана win_rate 0.463 и среднее около 0.458 то есть типичный командный сезон имеет долю побед около половины, а крайние значения (ниже 0.3 и выше 0.7) встречаются редко


## 3.2. Покажите изменение по годам: 8 баллов

Постройте линейный график медианы `win_rate` по годам из пункта 2.2.
        Подпишите оси. Найдите один год с заметно большей или меньшей медианой,
        чем у соседних доступных лет, и приведите числа. Отметьте в объяснении
        годы без данных.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("hockey_raw.csv")

by_year = df.groupby("year")["win_rate"].median()
plt.plot(by_year.index, by_year.values, marker="o")
plt.title("Медиана win_rate по годам")
plt.xlabel("Год (сезон)")
plt.ylabel("Медиана win_rate (доля побед)")
plt.show()


**Ответ 3.2:**

Линейный график медианы win_rate по годам показывает,что до 2003 года медиана колебалась в диапазоне примерно 0.42- 0.48,а с 2005 года поднялась до 0.49-0.52.Более заметный год 2005 медиана 0.512, что заметно выше предыдущего года 2003 (0.463).Год 2004 отсутствует в данных (сезон был отменён), поэтому на графике линия соединяет 2003 и 2005 напрямую, без точки за 2004.



## 3.3. Сравните голы и долю побед: 8 баллов

Постройте диаграмму рассеяния: `goal_diff` по горизонтали,
        `win_rate` по вертикали. Опишите общую связь и приведите две конкретные
        строки для иллюстрации. Можно ли по этому графику заключить, что знак
        `goal_diff` безошибочно определяет высокую долю побед?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("hockey_raw.csv")
plt.scatter(df["goal_diff"], df["win_rate"], alpha=0.5)
plt.title("Связь goal_diff и win_rate")
plt.xlabel("goal_diff (разница забитых и пропущенных шайб)")
plt.ylabel("win_rate (доля побед)")
plt.show()


**Ответ 3.3:**

Диаграмма рассеяния показывает положительную связь чем выше goal_diff,тем выше win_rate.Например, Calgary Flames 1990 с goal_diff = 81 имеют win_rate = 0.575,а Hartford Whalers 1990 с goal_diff =38 , win_rate = 0.388. НО по графику нельзя заключить, что знак goal_diff безошибочно определяет высокую долю побед точки разбросаны вокруг общей тенденции, и встречаются команды с положительным goal_diff, но не самой высокой win_rate.Связь — это тенденция,а не строгое правило.

## 3.4. Покажите историю выбранной команды: 7 баллов

Для команды из пункта 2.4 постройте график `win_rate` по годам.
        Укажите в ответе, есть ли годы без данных, и как вы показали их на графике.
        Найдите два соседних доступных сезона с заметным изменением
        и укажите значения в ответе.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("hockey_raw.csv")
team = "Boston Bruins"
team_df = df[df["team"] == team].sort_values("year")
plt.plot(team_df["year"], team_df["win_rate"], marker="o")
plt.title(f"win_rate команды {team} по годам")
plt.xlabel("Год (сезон)")
plt.ylabel("win_rate (доля побед)")
plt.show()

team_df["change"] = team_df["win_rate"].diff()
print(team_df[["year", "win_rate", "change"]])

**Ответ 3.4:**

График win_rate команды Boston Bruins по годам показан выше.Годы без данных есть: например, 2004 (сезон отменён из-за локаута) на графике линия соединяет соседние доступные точки длинным отрезком, что и показывает пропуск.Наиболее заметное изменение  между сезонами 1998 и 1999: win_rate изменилась с 0.476 до 0.293,разница ≈ 0.183

## 4.1. Проверьте последовательность лет: 5 баллов

Найдите все годы между минимальным и максимальным `year`,
        которых нет в таблице. Как отсутствие года может повлиять на линейный график?
        Сошлитесь на свой график из пункта 3.2.

In [ ]:
import pandas as pd

df = pd.read_csv("hockey_raw.csv")

all_years = set(range(df["year"].min(), df["year"].max() + 1))
present_years = set(df["year"].unique())
missing_years = sorted(all_years - present_years)

print("Минимальный год:", df["year"].min())
print("Максимальный год:", df["year"].max())
print("Отсутствующие годы:", missing_years)


**Ответ 4.1:**

Между минимальным 1990 и максимальным 2011 годами в таблице отсутствует один год  2004 (сезон был отменён из-за локаута).На линейном графике из пункта 3.2 это видно: линия соединяет точки 2003 медиана 0.463 и 2005 медиана 0.512 напрямую, «перепрыгивая» 2004. Отсутствие года может ввести в заблуждение: длинный отрезок между 2003 и 2005 выглядит как постепенное изменение за два сезона, хотя данных за 2004 нет вовсе.

## 4.2. Сравните число побед и долю: 6 баллов

Возьмите одну команду, которая есть в 1993, 1994 и 1995 годах.
        Выведите для неё `wins` и `win_rate` за эти годы.
        Какой вывод изменится, если смотреть только на число побед?
        Свяжите ответ с группировкой из пункта 2.2.

In [ ]:
import pandas as pd

df = pd.read_csv("hockey_raw.csv")
team = "Boston Bruins"
print(df[(df["team"] == team) & (df["year"].isin([1993, 1994, 1995]))][["team", "year", "wins", "win_rate"]])


**Ответ 4.2:**

Для Boston Bruins в 1993, 1994 и 1995 годах  wins = 42, 27 ,40, а win_rate = 0.5, 0.562 и 0.488. Если смотреть только на число побед, вывод будет таким: в 1994 команда сыграла заметно хуже 27 побед.Но по доле побед 1994 год — лучший из трёх (0.562). Это связано с группировкой из пункта 2.2: сезон 1994 был сокращён, поэтому число побед в нём несопоставимо с другими годами,а доля побед сопоставима. Вывод «команда ослабла в 1994» меняется на противоположный, если сравнивать долю побед, а не их число

## 4.3. Найдите собственное наблюдение: 5 баллов

Найдите ещё один интересный вам пример в таблице или на графике.
        Укажите команду, год, два значения и пункт, результат которого
        помог вам заметить пример. Как он уточняет ваш общий вывод?

In [ ]:
import pandas as pd
df = pd.read_csv("hockey_raw.csv")
print(df.nlargest(1, "win_rate")[["team", "year", "wins", "win_rate", "goal_diff"]])



**Ответ 4.3:**
пример: Detroit Red Wings в 1995 году: 62 победы и win_rate 0.756, при goal_diff +144. Это лучший сезон во всей таблице. Я заметила его благодаря пункту 2.3, где искались крайние результаты по win_rate, а также по диаграмме рассеяния из пункта 3.3  это точка в правом верхнем углу.Пример уточняет общий вывод: рекордная доля побед сопровождается рекордной разницей голов,что подтверждает положительную связь из пункта 3.3, но такой сезон — редкий выброс, тогда как типичный сезон (пункт 2.1) имеет win_rate около 0.5.

## 4.4. Сформулируйте выводы: 4 баллов

Сформулируйте два вывода из своей работы. Для каждого приведите
        номер таблицы или графика и одно число. Назовите одно ограничение данных,
        которое важно помнить при чтении результатов.

In [ ]:
# 4.4. Ваш код
# Добавьте ячейки при необходимости.

**Ответ 4.4:**

Вывод 1:Доля побед команд NHL стабильна во времени: медиана win_rate по годам колеблется в узком диапазоне примерно 0.42–0.52(таблица из пункта 2.2, например, максимум 0.524 в 2010 году), тогда как число побед растёт из-за увеличения числа матчей в сезоне.Поэтому для сравнения сезонов между собой нужно использовать win_rate,а не wins.

Вывод 2: Между разницей забитых и пропущенных шайб (goal_diff) и долей побед есть положительная связь (график из пункта 3.3): например, Detroit Red Wings в 1995 году имели goal_diff +144 и win_rate 0.756 — лучший сезон в таблице. Однако это тенденция, а не строгое правило: встречаются команды с положительным goal_diff, но невысокой долей побед (например, Buffalo Sabres 1990: +14 и 0.388).

Ограничение данных. В таблице отсутствует 2004 год (сезон был отменён из-за локаута), поэтому на линейных графиках линия соединяет 2003 и 2005 напрямую, что может искажать восприятие динамики. Кроме того, в ранних сезонах есть пропуски в ot_losses, поэтому этот столбец нельзя использовать для тех лет.